In [ ]:
import frust as ft

wf = ft.workflows.catalyst_screen(
    csv_path="../datasets/screen.csv",
    screening="gxtb-default",
    level="full",  # use "low_cost" or "dft_ranked" for ΔE-only screens
    method="r2scan-3c",
    ranking_solvation="method",  # inherits SMD chloroform here
    scope="barriers",  # use "full_cycle" for the balanced profile
)

In [ ]:
display(wf.plan())
display(wf.show_stages(execution="dft_staged"))

preview = wf.preview(targets=[0], n_confs=1)
ft.plot_mols(preview)

In [ ]:
from frust.cluster import ClusterConfig, Resources

cluster = ClusterConfig(backend="slurm", partition="kemi1", log_dir="logs")
resources = {
    "init": Resources(cpus=12, mem_gb=12, timeout_min=7200),
    "dft_opt": Resources(cpus=12, mem_gb=12, timeout_min=7200),
    "dft_hessian": Resources(cpus=12, mem_gb=24, timeout_min=7200),
    "dft_ts_opt": Resources(cpus=12, mem_gb=12, timeout_min=7200),
    "dft_freq": Resources(cpus=12, mem_gb=24, timeout_min=7200),
    "dft_solv_sp": Resources(cpus=12, mem_gb=12, timeout_min=7200),
}

submission = wf.submit(
    out_dir="results",
    cluster=cluster,
    execution="dft_staged",
    stage_resources=resources,
)
submission

In [ ]:
# Run after finalization, either on the cluster or after downloading results/.
run = ft.screen.open_run("results")
display(run.summary())
display(run.barriers())
display(run.review_queue())